# 3.2 Demo - File Explosion
**Lesson:** Introduction to Designing the Foundation

A table is only as fast as its **physical layout**. The most common layout bug is the *small-file problem* ("file explosion"):
- frequent tiny appends (streaming with short triggers, row-by-row inserts)
- **over-partitioning**: `PARTITIONED BY` a high-cardinality column
- writing with too many partitions (`repartition(10000)`)

Every file costs a listing entry, a metadata entry, a task, and an open/seek. Thousands of KB-sized files can make a query 10-100x slower than the same data in a few well-sized files (Databricks targets **~128 MB - 1 GB** per file).

In [ ]:
from adelab.classroom import setup
from adelab import perf
from pyspark.sql import functions as F

DA = setup("3.2", reset=True)

def transactions(n=2_000_000):
    return (spark.range(n)
            .withColumn("customer_id", (F.rand(seed=1) * 50_000).cast("int"))
            .withColumn("store_id", (F.rand(seed=2) * 200).cast("int"))
            .withColumn("amount", F.round(F.rand(seed=3) * 500, 2))
            .withColumn("ts", F.expr("timestamp'2026-01-01' + make_interval(0,0,0,0,0,0, id % 31536000)")))

## A. Baseline: a well-sized table

In [ ]:
transactions().coalesce(4).write.saveAsTable("tx_good")
display(perf.table_files(spark, "tx_good"))

## B. Explosion #1: many tiny files
Same data, written as 1,000 files. This is what you get from many small appends or an aggressive `repartition`.

![1,000 tiny files spend most scan time on per-file overhead; 4 right-sized files spend it reading data](../images/3.2-small-vs-right-sized-files.svg)

**How to read it:** each red block is fixed work per file (listing, opening, reading the footer, scheduling a task); blue is reading rows.
`tx_tiny_files` pays the red block 1,000 times for the same rows that `tx_good` stores in 4 files.
Compare `numFiles` and `avgFileKB` of the two `table_files` outputs.

In [ ]:
transactions().repartition(1000).write.saveAsTable("tx_tiny_files")
display(perf.table_files(spark, "tx_tiny_files"))

## C. Explosion #2: over-partitioning
Partitioning creates one directory per distinct value, and every write task produces a file in **each** directory it touches. Here 400 k rows are `PARTITIONED BY (store_id)`: 200 directories with a few KB per file, for a table of a few MB. With a high-cardinality column such as `customer_id` (50,000 values) it would be 50,000 directories.

Partition only by **low-cardinality** columns that are almost always filtered on, with at least ~1 GB per partition. On Databricks, prefer **Liquid Clustering** (lesson 3.3) for new tables.

![PARTITIONED BY store_id creates one directory per value and a small file per write task in each directory](../images/3.2-over-partitioning.svg)

**How to read it:** the number of files is roughly (write tasks) × (distinct partition values), regardless of how little data each holds.
`table_files` below shows the result for `tx_over_partitioned`: many files, a few KB each.

In [ ]:
(transactions(400_000).write.partitionBy("store_id").saveAsTable("tx_over_partitioned"))
display(perf.table_files(spark, "tx_over_partitioned"))

## D. Measure
The same aggregate on each table. Watch `files_read` and the time. Then open the **SQL** tab in the Spark UI and compare the *scan time* and *number of files read* of the three scans.

In [ ]:
q = "SELECT store_id, count(*) AS n, sum(amount) AS revenue FROM {t} GROUP BY store_id"
results = [perf.run(spark.sql(q.format(t=t)), t) for t in ("tx_good", "tx_tiny_files", "tx_over_partitioned")]
display(perf.compare(*results))
perf.plot(results, "files_read", "Files read by the same aggregate", highlight="tx_good", note="well-sized files");
perf.plot(results, "seconds", "Query time", highlight="tx_good", note="well-sized files");

Compare the `tx_tiny_files` and `tx_over_partitioned` bars with `tx_good`: same rows, many more files to open, and the time follows the file count.

## E. Fix: compaction with `OPTIMIZE`
`OPTIMIZE` rewrites small files into large ones (bin-packing). The old files stay until `VACUUM`, so time travel still works.

![OPTIMIZE reads the small files and writes a few large ones in a new table version; the transaction log records the swap](../images/3.2-optimize-compaction.svg)

**How to read it:** the rows don't change, only the files that hold them. The commit in `_delta_log` removes the small files and adds
the large ones, which is why `numFiles` drops in the next output while older versions stay readable until `VACUUM`.

In [ ]:
display(spark.sql("OPTIMIZE tx_tiny_files"))
display(perf.table_files(spark, "tx_tiny_files"))
display(perf.compare(perf.run(spark.sql(q.format(t="tx_tiny_files")), "tx_tiny_files after OPTIMIZE")))

## F. Prevent it: optimized writes and auto compaction
| Setting | Effect |
|---|---|
| `delta.autoOptimize.optimizeWrite = true` | Shuffles data before writing so each partition writes fewer, larger files |
| `delta.autoOptimize.autoCompact = true` | After a write, compacts small files synchronously if there are enough of them |
| Predictive optimization (Databricks, UC managed tables) | Runs OPTIMIZE / VACUUM / ANALYZE automatically. Usually the best choice |

Simulate 30 tiny appends, with and without auto compaction:

In [ ]:
for name, props in (("tx_appends_plain", ""),
                    ("tx_appends_autocompact", "TBLPROPERTIES ('delta.autoOptimize.autoCompact' = 'true', "
                                               "'delta.autoOptimize.optimizeWrite' = 'true')")):
    spark.sql(f"CREATE TABLE {name} (id BIGINT, customer_id INT, store_id INT, amount DOUBLE, ts TIMESTAMP) {props}")
    batch = transactions(5_000).repartition(4)
    for _ in range(30):
        batch.write.mode("append").saveAsTable(name)
    print(perf.table_files(spark, name))

## G. Key takeaways
- Check `numFiles` and the average file size (`DESCRIBE DETAIL`) of hot tables regularly.
- Never partition by high-cardinality columns. Use Liquid Clustering instead.
- Batch streaming writes with sensible triggers. Turn on optimized writes and auto compaction, or predictive optimization.
- `OPTIMIZE` fixes existing tables. `VACUUM` removes the replaced files after the retention period.

In [ ]:
DA.cleanup()